<a href="https://colab.research.google.com/github/heyiamheshan/ASR-Nemotron-3.5-ASR-NVIDIA/blob/main/Sinhala_ASR_New.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!nvidia-smi

Thu Jun 11 05:03:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os
PROJECT = '/content/drive/MyDrive/sinhala_asr'
os.makedirs(PROJECT, exist_ok=True)

# verify we can write to it
with open(f'{PROJECT}/test.txt', 'w') as f:
    f.write('drive works')
print("Drive folder ready:", os.listdir(PROJECT))

Drive folder ready: ['test.txt']


In [5]:
import os
os.makedirs('/content/data', exist_ok=True)
%cd /content/data

# 1. transcripts (small)
!wget -q https://www.openslr.org/resources/52/utt_spk_text.tsv
print("transcripts downloaded")

# 2. one audio zip part (~900MB)
!wget -q --show-progress https://www.openslr.org/resources/52/asr_sinhala_0.zip
print("audio zip downloaded")

# 3. unzip
!unzip -q asr_sinhala_0.zip
print("unzipped")

# 4. see what we got
!ls /content/data

/content/data
transcripts downloaded
asr_sinhala_0.zip   100%[===================>] 872.84M  7.75MB/s    in 2m 14s  
audio zip downloaded
unzipped
asr_sinhala  asr_sinhala_0.zip	utt_spk_text.tsv


In [6]:
import os
# how is the audio organized?
root = '/content/data/asr_sinhala'
print(os.listdir(root)[:5])
sub = os.path.join(root, os.listdir(root)[0])
if os.path.isdir(sub):
    print(os.listdir(sub)[:5])

['data', 'utt_spk_text.tsv', 'LICENSE']
['05', '03', '00', '02', '0e']


In [7]:
import os, json, random
!pip install -q soundfile
import soundfile as sf

AUDIO_ROOT = '/content/data/asr_sinhala'
TSV = '/content/data/utt_spk_text.tsv'

# index all audio files we actually have (part 0 only)
audio_index = {}
for dirpath, _, files in os.walk(AUDIO_ROOT):
    for f in files:
        if f.endswith('.flac') or f.endswith('.wav'):
            audio_index[os.path.splitext(f)[0]] = os.path.join(dirpath, f)
print(f"audio files found: {len(audio_index)}")

# read TSV and keep rows whose audio we have
rows = []
with open(TSV, encoding='utf-8') as f:
    for line in f:
        parts = line.rstrip('\n').split('\t')
        if len(parts) >= 3:
            fid, _, text = parts[0], parts[1], parts[2]
            if fid in audio_index and text.strip():
                rows.append((audio_index[fid], text.strip()))
print(f"matched audio+transcript pairs: {len(rows)}")

# shuffle and split: 95% train, 5% test
random.seed(42); random.shuffle(rows)
split = int(len(rows) * 0.95)
splits = {'train': rows[:split], 'test': rows[split:]}

for name, data in splits.items():
    out_path = f'/content/{name}_manifest.json'
    written = 0
    with open(out_path, 'w', encoding='utf-8') as out:
        for path, text in data:
            try:
                info = sf.info(path)
                dur = info.frames / info.samplerate
            except Exception:
                continue
            out.write(json.dumps({
                "audio_filepath": path, "duration": round(dur, 2),
                "text": text, "target_lang": "si-LK",
            }, ensure_ascii=False) + "\n")
            written += 1
    print(f"{name}: wrote {written}")

print("\nsample line:")
print(open('/content/train_manifest.json', encoding='utf-8').readline())

audio files found: 11550
matched audio+transcript pairs: 11550
train: wrote 10972
test: wrote 578

sample line:
{"audio_filepath": "/content/data/asr_sinhala/data/03/0360dc5ae2.flac", "duration": 2.6, "text": "wood", "target_lang": "si-LK"}



In [8]:
!pip install -q "nemo_toolkit[asr] @ git+https://github.com/NVIDIA/NeMo.git"
!pip install -q "numpy==1.26.4"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 903.9/903.9 kB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 811.0/811.0 kB 60.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 90.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 64.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 54.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.3/233.3 kB 17.6 MB/s eta 0

In [1]:
import nemo.collections.asr as nemo_asr
print("NeMo OK")

NeMo OK


In [2]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import nemo.collections.asr as nemo_asr
from omegaconf import OmegaConf

# 1. load base model (~2.4 GB download, a minute or two)
model = nemo_asr.models.ASRModel.from_pretrained("nvidia/nemotron-3.5-asr-streaming-0.6b")
print("model loaded")

# 2. attach data
vsrc = model.cfg.validation_ds
def make_cfg(manifest, shuffle):
    return OmegaConf.create({
        "manifest_filepath": manifest, "sample_rate": 16000, "batch_size": 1,
        "shuffle": shuffle, "num_workers": 2, "pin_memory": True,
        "use_lhotse": True, "use_bucketing": False, "is_tarred": False,
        "shard_manifests": False, "max_duration": 16.0, "min_duration": 0.1,
        "prompt_field": "target_lang", "lang_field": "target_lang",
        "prompt_dictionary": vsrc.prompt_dictionary,
        "num_prompts": vsrc.get('num_prompts', 128),
        "subsampling_factor": vsrc.get('subsampling_factor', 8),
        "batch_duration": None,
    })

model.setup_training_data(make_cfg('/content/train_manifest.json', True))
model.setup_validation_data(make_cfg('/content/test_manifest.json', False))
print("data attached")

# 3. freeze encoder — train only decoder + joint + prompt (~29M params)
model.encoder.freeze()
for p in model.encoder.parameters():
    p.requires_grad = False
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"trainable params: {trainable/1e6:.1f}M (encoder frozen)")

nemotron-3.5-asr-streaming-0.6b.nemo:   0%|          | 0.00/2.37G [00:00<?, ?B/s]

[NeMo I 2026-06-11 05:29:20 mixins:184] Tokenizer SentencePieceTokenizer initialized with 13087 tokens


[NeMo W 2026-06-11 05:29:25 modelPT:288] You tried to register an artifact under config key=tokenizer.model_path but an artifact for it has already been registered.
[NeMo W 2026-06-11 05:29:25 modelPT:288] You tried to register an artifact under config key=tokenizer.vocab_path but an artifact for it has already been registered.


[NeMo I 2026-06-11 05:29:25 mixins:184] Tokenizer SentencePieceTokenizer initialized with 13087 tokens


[NeMo W 2026-06-11 05:29:31 modelPT:176] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: null
    sample_rate: 16000
    use_lhotse: true
    shard_manifests: true
    batch_duration: 200
    quadratic_duration: 15
    num_buckets: 30
    shuffle: true
    num_workers: 8
    pin_memory: true
    max_duration: 20
    min_duration: 0.1
    is_tarred: true
    tarred_audio_filepaths: null
    shuffle_n: 2048
    slice_length: 100
    bucketing_strategy: fully_randomized
    bucketing_batch_size: null
    bucket_buffer_size: 10000
    shuffle_buffer_size: 10000
    prompt_field: target_lang
    prompt_dictionary:
      en-US: 0
      en: 0
      en-GB: 1
      enGB: 1
      es-ES: 2
      esES: 2
      es-US: 3
      es: 3
      zh-CN: 4
      zh-ZH: 4
      zh-TW: 5
      hi-IN: 6
      hi: 6
      hi-HI: 6
      ar-AR: 7
     

[NeMo I 2026-06-11 05:29:40 rnnt_models:226] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.005, 'clamp': -1.0}
[NeMo I 2026-06-11 05:29:40 rnnt_models:226] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.005, 'clamp': -1.0}
[NeMo I 2026-06-11 05:29:40 rnnt_models:226] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.005, 'clamp': -1.0}
[NeMo I 2026-06-11 05:29:40 rnnt_bpe_models_prompt:146] Model with prompt feature has been initialized (RNNT-only)
[NeMo I 2026-06-11 05:29:52 save_restore_connector:286] Model EncDecRNNTBPEModelWithPrompt was successfully restored from /root/.cache/huggingface/hub/models--nvidia--nemotron-3.5-asr-streaming-0.6b/snapshots/24b151a851dd15909e1fc611b11bb2da52b9fc81/nemotron-3.5-asr-streaming-0.6b.nemo.
model loaded
[NeMo I 2026-06-11 05:29:53 audio_to_text_lhotse_prompt_index:86] LhotseSpeechToTextBpeDatasetWithPromptIndex: default_prompt_mo

[NeMo W 2026-06-11 05:29:54 dataloader:881] The following configuration keys are ignored by Lhotse dataloader: prompt_field,subsampling_factor,prompt_dictionary,is_tarred,shard_manifests,num_prompts
[NeMo W 2026-06-11 05:29:54 dataloader:533] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)


[NeMo I 2026-06-11 05:29:54 dataloader:659] Creating a Lhotse DynamicCutSampler (bucketing is disabled, (max_batch_duration=None max_batch_size=1)
[NeMo I 2026-06-11 05:29:55 audio_to_text_lhotse_prompt_index:86] LhotseSpeechToTextBpeDatasetWithPromptIndex: default_prompt_mode=unified, unified_auto_ratio=0.5
[NeMo I 2026-06-11 05:29:55 rnnt_bpe_models_prompt:181] Setting up Lhotse dataset with prompt index support (RNNT-only model creates prompt tensors)
[NeMo I 2026-06-11 05:29:55 dataloader:342] We will be using a Lhotse DataLoader.


[NeMo W 2026-06-11 05:29:55 dataloader:881] The following configuration keys are ignored by Lhotse dataloader: prompt_field,subsampling_factor,prompt_dictionary,is_tarred,shard_manifests,num_prompts
[NeMo W 2026-06-11 05:29:55 dataloader:533] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)


[NeMo I 2026-06-11 05:29:55 dataloader:659] Creating a Lhotse DynamicCutSampler (bucketing is disabled, (max_batch_duration=None max_batch_size=1)
data attached
trainable params: 28.9M (encoder frozen)


In [ ]:
import lightning.pytorch as pl
from lightning.pytorch.callbacks import Callback
from omegaconf import open_dict

# optimizer: gentle LR, no scheduler
with open_dict(model.cfg):
    model.cfg.optim.lr = 1e-4
    if 'sched' in model.cfg.optim:
        del model.cfg.optim.sched

DRIVE = '/content/drive/MyDrive/sinhala_asr'

# custom callback: save .nemo to Drive periodically (survives disconnects)
class DriveSaver(Callback):
    def __init__(self, every_n_steps=2000):
        self.every = every_n_steps
    def on_train_batch_end(self, trainer, pl_module, *args, **kwargs):
        step = trainer.global_step
        if step > 0 and step % self.every == 0:
            path = f'{DRIVE}/sinhala_step{step}.nemo'
            pl_module.save_to(path)
            print(f"\n[saved checkpoint to Drive at step {step}]")

trainer = pl.Trainer(
    devices=1, accelerator='gpu',
    max_steps=11000,                # ~1 epoch
    limit_val_batches=0, num_sanity_val_steps=0,
    enable_checkpointing=False, logger=False,
    precision=32, gradient_clip_val=1.0,
    log_every_n_steps=50,
    callbacks=[DriveSaver(every_n_steps=2000)],
)
model.set_trainer(trainer)

print("TRAINING STARTED — this runs for ~2-3 hours")
trainer.fit(model)
print("TRAINING DONE")

# final save to Drive
model.save_to(f'{DRIVE}/sinhala_final_epoch1.nemo')
print("final model saved to Drive")

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs


TRAINING STARTED — this runs for ~2-3 hours


INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


[NeMo I 2026-06-11 05:34:07 modelPT:782] Optimizer config = AdamW (
    Parameter Group 0
        amsgrad: False
        betas: (0.9, 0.98)
        capturable: False
        decoupled_weight_decay: True
        differentiable: False
        eps: 1e-08
        foreach: None
        fused: None
        lr: 0.0001
        maximize: False
        weight_decay: 0.001
    )
[NeMo I 2026-06-11 05:34:07 lr_scheduler:844] Scheduler not initialized as no `sched` config supplied to setup_optimizer()


INFO: 
  | Name              | Type                              | Params | Mode 
--------------------------------------------------------------------------------
0 | preprocessor      | AudioToMelSpectrogramPreprocessor | 0      | train
1 | encoder           | ConformerEncoder                  | 609 M  | eval 
2 | decoder           | RNNTDecoder                       | 14.9 M | train
3 | joint             | RNNTJoint                         | 9.5 M  | train
4 | loss              | RNNTLoss                          | 0      | train
5 | spec_augmentation | SpectrogramAugmentation           | 0      | train
6 | wer               | WER                               | 0      | train
7 | prompt_kernel     | Sequential                        | 4.5 M  | train
--------------------------------------------------------------------------------
28.9 M    Trainable params
609 M     Non-trainable params
637 M     Total params
2,551.988 Total estimated model params size (MB)
24        Modules in train

...

[NeMo I 2026-06-11 05:45:27 wer:318] 
    
[NeMo I 2026-06-11 05:45:27 wer:319] WER reference: ⁇ ,  ⁇ ,  ⁇   ⁇   ⁇   ⁇ 
[NeMo I 2026-06-11 05:45:27 wer:320] WER predicted:⁇  ⁇ 
[NeMo I 2026-06-11 05:45:31 wer:318] 
    
[NeMo I 2026-06-11 05:45:31 wer:319] WER reference: ⁇   ⁇   ⁇   ⁇   ⁇   ⁇   ⁇ .
[NeMo I 2026-06-11 05:45:31 wer:320] WER predicted:⁇  ⁇ 
[NeMo I 2026-06-11 05:45:35 wer:318] 
    
[NeMo I 2026-06-11 05:45:35 wer:319] WER reference: ⁇   ⁇   ⁇   ⁇   ⁇ 
[NeMo I 2026-06-11 05:45:35 wer:320] WER predicted:⁇  ⁇  ⁇ 
[NeMo I 2026-06-11 05:45:41 wer:318] 
    
[NeMo I 2026-06-11 05:45:41 wer:319] WER reference: ⁇   ⁇   ⁇ 
[NeMo I 2026-06-11 05:45:41 wer:320] WER predicted:⁇  ⁇ 
[NeMo I 2026-06-11 05:45:45 wer:318] 
    
[NeMo I 2026-06-11 05:45:45 wer:319] WER reference: ⁇   ⁇   ⁇   ⁇   ⁇   ⁇   ⁇ 
[NeMo I 2026-06-11 05:45:45 wer:320] WER predicted:⁇  ⁇  ⁇  ⁇ 
[NeMo I 2026-06-11 05:45:49 wer:318] 
    
[NeMo I 2026-06-11 05:45:49 wer:319] WER reference: ⁇   ⁇   ⁇   ⁇ 
[NeMo I 202

[NeMo W 2026-06-11 05:45:50 nemo_logging:364] /usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 70 will likely result in GPU under-utilization due to low occupancy.
      warn(errors.NumbaPerformanceWarning(msg))
    


[NeMo I 2026-06-11 05:45:54 wer:318] 
    
[NeMo I 2026-06-11 05:45:54 wer:319] WER reference: ⁇   ⁇   ⁇ 
[NeMo I 2026-06-11 05:45:54 wer:320] WER predicted:
[NeMo I 2026-06-11 05:45:58 wer:318] 
    
[NeMo I 2026-06-11 05:45:58 wer:319] WER reference: ⁇   ⁇   ⁇   ⁇   ⁇   ⁇ 
[NeMo I 2026-06-11 05:45:58 wer:320] WER predicted:
[NeMo I 2026-06-11 05:46:02 wer:318] 
    
[NeMo I 2026-06-11 05:46:02 wer:319] WER reference: ⁇   ⁇   ⁇   ⁇ 
[NeMo I 2026-06-11 05:46:02 wer:320] WER predicted:⁇  ⁇  ⁇  ⁇ 
[NeMo I 2026-06-11 05:46:07 wer:318] 
    
[NeMo I 2026-06-11 05:46:07 wer:319] WER reference: ⁇   ⁇   ⁇   ⁇   ⁇   ⁇   ⁇ 
[NeMo I 2026-06-11 05:46:07 wer:320] WER predicted:
[NeMo I 2026-06-11 05:46:12 wer:318] 
    
[NeMo I 2026-06-11 05:46:12 wer:319] WER reference: ⁇   ⁇   ⁇   ⁇ 
[NeMo I 2026-06-11 05:46:12 wer:320] WER predicted:
[NeMo I 2026-06-11 05:46:16 wer:318] 
    
[NeMo I 2026-06-11 05:46:16 wer:319] WER reference: ⁇   ⁇   ⁇   ⁇ 
[NeMo I 2026-06-11 05:46:16 wer:320] WER predicted:⁇ 

[NeMo W 2026-06-11 05:46:38 nemo_logging:364] /usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 95 will likely result in GPU under-utilization due to low occupancy.
      warn(errors.NumbaPerformanceWarning(msg))
    


[NeMo I 2026-06-11 05:46:39 wer:318] 
    
[NeMo I 2026-06-11 05:46:39 wer:319] WER reference: ⁇   ⁇   ⁇   ⁇   ⁇ 
[NeMo I 2026-06-11 05:46:39 wer:320] WER predicted:⁇  ⁇ 
[NeMo I 2026-06-11 05:46:43 wer:318] 
    
[NeMo I 2026-06-11 05:46:43 wer:319] WER reference: ⁇   ⁇   ⁇   ⁇   ⁇ 
[NeMo I 2026-06-11 05:46:43 wer:320] WER predicted:⁇  ⁇  ⁇  ⁇ 
[NeMo I 2026-06-11 05:46:48 wer:318] 
    
[NeMo I 2026-06-11 05:46:48 wer:319] WER reference:dc to dc converter circuits
[NeMo I 2026-06-11 05:46:48 wer:320] WER predicted:⁇  ⁇ 
[NeMo I 2026-06-11 05:46:52 wer:318] 
    
[NeMo I 2026-06-11 05:46:52 wer:319] WER reference: ⁇   ⁇   ⁇ 
[NeMo I 2026-06-11 05:46:52 wer:320] WER predicted:⁇  ⁇  ⁇ 
[NeMo I 2026-06-11 05:46:57 wer:318] 
    
[NeMo I 2026-06-11 05:46:57 wer:319] WER reference: ⁇   ⁇   ⁇   ⁇ .
[NeMo I 2026-06-11 05:46:57 wer:320] WER predicted:⁇  ⁇ 
[NeMo I 2026-06-11 05:47:02 wer:318] 
    
[NeMo I 2026-06-11 05:47:02 wer:319] WER reference: ⁇   ⁇   ⁇   ⁇   ⁇ 
[NeMo I 2026-06-11 05:47: